In [1]:
%load_ext autoreload
%autoreload 2

In [38]:
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from lib.data.datasets import AccRawDataset
from lib.data.dataloading import load_raw, load_nursing_5_class
from lib.config import RAW_DIR
from lib.modules import optimization_loop_xonly,sample_regnet, optimization_loop_multi_class
from lib.models import RegNetv3, RegNetMAEv3, CosineMSELoss, RegNetv3Ci
from pathlib import Path
import json

In [39]:
CONFIG = {
    'WINDOW_SIZE':2001,
    'WINDOW_STRIDE':2001 // 16,
    'NURSING_STRIDE': 2001 // 16,
    'BATCH_SIZE': 128,
    'LEARNING_RATE': 1e-3,
    'CLASS_LR': 3e-4,
    'ENC_LEARNING_RATE': 5e-5,
    'TEST_SIZE': 0.1,
    'NURSING_TEST_SIZE': 0.25,
    'DEVICE': 'cuda:1',
    'DEPTHI': [2],
    'WIDTHI': [64],
    'NTL': 1,
    'DMODEL': 0,
    'MASKPCT': 0.5,
    'PDROPOUT': 0.0,
    'FREEZE': False,
    'WEIGHTS_FILE': None,
    'LSTM_SEQLEN': 3,
}
CONFIG['PRETRAINED'] = bool(CONFIG['WEIGHTS_FILE'])

nursing_trainloader, nursing_testloader = load_nursing_5_class(
    range(11,71), 
    CONFIG['WINDOW_SIZE']*CONFIG['LSTM_SEQLEN'], 
    test_size=CONFIG['NURSING_TEST_SIZE'], 
    batch_size=CONFIG['BATCH_SIZE'],
    stride=CONFIG['NURSING_STRIDE'],
)

In [87]:
class ClassifierLSTM(nn.Module):
    def __init__(self, CONFIG):
        super().__init__()
        self.weights_file = CONFIG.get('CLASS_WEIGHTS_FILE', None)
        if not self.weights_file:
            raise ValueError('No weights file provided')
        self.winsize = CONFIG['WINDOW_SIZE']
        self.seq_len = CONFIG['LSTM_SEQLEN']

        self.regnet = RegNetv3Ci(CONFIG=CONFIG)
        print(f'Loading weights from {self.weights_file}')
        self.regnet.load_state_dict(torch.load(self.weights_file))
        for p in self.regnet.parameters():
            p.requires_grad = False
        
        self.lstm = nn.LSTM(
            input_size=5,   # number of classes
            hidden_size=64,
            num_layers=1,
            batch_first=True,
        )

        self.out = nn.Linear(64, 5)
        
    def forward(self, x):
        x = x.view(x.shape[0], self.seq_len, 3, self.winsize)
        ys = []
        for i in range(self.seq_len):
            ys.append(self.regnet(x[:,i]).unsqueeze(1))
        ys = torch.cat(ys,dim=1)
        o, (h,c) = self.lstm(ys)
        x = self.out(o[:,-1])
        return x

In [90]:
model_dir = Path('/home/musa/eating-detection/dev/9_regnet-mae/random-search-class-fixed/[4, 13, 3]-[48, 120, 304]-pretrained-ci')
CONFIG = json.load(open(model_dir / 'config.json'))
CONFIG['LSTM_SEQLEN'] = 3
CONFIG['CLASS_WEIGHTS_FILE'] = str(model_dir / 'best_model.pt')

model = ClassifierLSTM(CONFIG).to(CONFIG['DEVICE'])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['CLASS_LR'])

for X,y in nursing_trainloader:
    X = X.to(CONFIG['DEVICE'])
    y = y.to(CONFIG['DEVICE'])
    logits = model(X)
    break

lstm_outdir = str(model_dir).replace('random-search-class-fixed', 'random-search-lstm')
optimization_loop_multi_class(
    model,
    nursing_trainloader,
    nursing_testloader,
    criterion,
    optimizer,
    epochs=500,
    device=CONFIG['DEVICE'],
    patience=50,
    outdir=lstm_outdir,
    writer=lstm_outdir,
    config=CONFIG
)

latent dim: 125
Loading weights from /home/musa/eating-detection/dev/9_regnet-mae/random-search-class-fixed/[4, 13, 3]-[48, 120, 304]-pretrained-ci/best_model.pt


: Epoch 1: Train Loss: 1.0753: Dev Loss: 1.0778, Dev F1: 0.53855:   0%|          | 2/500 [01:13<5:05:17, 36.78s/it]